## Procesamiento del Resultado de la API

En esta sección se normaliza la información obtenida desde la API para transformarla en un formato consistente y dejarla estructurada de la misma manera que el *dataframe* principal.


In [31]:
import json
from pathlib import Path
import pandas as pd

PATH = './scopus_results_fieldv2.json'

In [73]:
fields = {
    'authors_info': 'Authors',
    'authors_info_with_id': 'Author full names',
    'authors_id': 'Author(s) ID',
    'dc:title': 'Title',
    'prism:coverDate': 'Date',
    'prism:volume': 'Volume',
    'prism:issueIdentifier': 'Issue',
    'citedby-count': 'Cited by',
    'prism:doi': 'DOI',
    'subtypeDescription': 'Document Type'
}

## __Necesito__

| Clave                  | Descripción (general) |
|-------------------------|-----------------------|
| `dc:title`            | Título del artículo/documento |
| `prism:publicationName`| Nombre de la revista o fuente |
| `prism:volume`        | Volumen de la publicación |
| `prism:issueIdentifier`| Número de la edición (issue) |
| `prism:pageRange`     | Páginas del artículo |
| `prism:coverDate`     | Fecha de publicación (ISO: YYYY-MM-DD) |
| `prism:doi`           | DOI del artículo |
| `citedby-count`       | Número de citas |
| `subtypeDescription`   | Descripción del subtipo (ej. Article, Review) |
| `author`               | Lista de autores |


In [98]:
class Elsjson: 
    def __init__(self, path = ''):
        self.path = Path(path) if path else None
        self.data = None

        self.columns = []

    @property
    def path(self):
        """ Gets the path"""
        if not self._path:
            raise AttributeError("path has not been loaded yet. Call self.path = Path(...)")
        return self._path
    
    @path.setter
    def path(self, path):
        """ Sets the path"""
        self._path = path

    @property
    def data(self):
        """Returns the parsed JSON data as a Python object (dict or list)."""
        if not self._data:
            raise AttributeError("JSON data has not been loaded yet. Call readJSON() first.")
        return self._data

    @data.setter
    def data(self, data):
        """Stores the JSON data (as a Python dict, list, or raw JSON string)."""
        self._data = data

    @property
    def entries(self):
        return self.data['search-results']['entry']

    def readJSON(self):
        """
        Reads the JSON file from the local path (used only when the file 
        is stored locally instead of in a database).
        """
        with self.path.open('r', encoding = 'utf-8') as f:
            self.data = json.load(f)

    def preprocess_author(self, authors):
        return ';'.join(f'{auth["authname"]}' for auth in authors)

    def preprocess_author_with_id(self, authors):
        return ';'.join(f'{auth["authname"]} ({auth["authid"]})' for auth in authors)
    
    def preprocess_author_id(self, authors):
        return ';'.join(f'{auth["authid"]}' for auth in authors)


    def preprocess(self):
        for entry in self.entries:
            entry['authors_info'] = self.preprocess_author(entry['author'])
            entry['authors_info_with_id'] = self.preprocess_author_with_id(entry['author'])
            entry['authors_id'] = self.preprocess_author_id(entry['author'])

    def execute(self):
        try:    
            self.preprocess() #Procesamos y colocamos la key de author
            df = pd.json_normalize(self.entries)

            # modificamos un poco el df
            columnas_a_conservar = [c for c in df.columns if c in fields]
            df = df[columnas_a_conservar].rename(columns=fields)

            return df
        except AttributeError as e:
            print(e)

In [99]:
prueba = Elsjson(PATH)
prueba.readJSON()


In [100]:
prueba.execute().head(1)

,Title,Volume,Date,DOI,Cited by,Document Type,Authors,Author full names,Author(s) ID,Issue
0,Biogeochemical study of the periglacial slopes...,443,2026-01-01,10.1016/j.icarus.2025.116783,0,Article,Leal M.A.;Tovar D.;de Pablo M.A.;Bonilla M.A.;...,Leal M.A. (58310191500);Tovar D. (58310000000)...,58310191500;58310000000;7003652519;60054445900...,NaN
